# 47 - Encode the scaled corpus with Linq-Embed-Mistral

Same checkpoint-and-resume pattern as `20_baseline_linq_mistral.ipynb`, this model is the slow one: a 7B-parameter model loaded in fp16, and on this cluster's storage, just *loading* the model can take 20-27 minutes, on top of whatever the actual per-job SLURM wall-time limit is (30 min in the original run). That notebook's own comment says as much, and the original 98,716-company run took roughly 34 minutes of actual encoding time (measured), spread across several resubmitted jobs because so little of each 30-minute job was left after loading.

**This run is about 4x the size (~397K companies vs 98,716), so expect roughly 4x the encoding time, on the order of 2-2.5 hours of actual encoding, split across many more resubmitted `run.sh` jobs than the original run needed.** There is no way around this without a longer job wall-time allocation, if your cluster allows requesting a longer single job, that would cut the number of resubmissions down substantially (most of the wasted time is model *loading*, not encoding). Otherwise, just keep resubmitting, the checkpoint means no progress is ever lost, each run picks up exactly where the last one stopped.

In [ ]:
import time
SCRIPT_START = time.time()  # marks total job elapsed time, used to stop encoding safely before the SLURM wall time

import sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

RESULT_DIR = Path("result/47_encode_linq_mistral_scaled")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[Setup] Result folder : {RESULT_DIR}/ -- ready")

combined = pd.read_parquet("result/44_build_scaled_corpus/combined_pool.parquet")
rich_texts = combined["rich_text"].tolist()
print(f"[Load] Companies to encode: {len(rich_texts):,}")

print(f"[GPU] CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[GPU] Device : {torch.cuda.get_device_name(0)}")
    print(f"[GPU] VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    DEVICE = "cuda"
else:
    print("[GPU] WARNING: No GPU -- this model needs a GPU to be practical")
    DEVICE = "cpu"

In [ ]:
import os

MAX_LENGTH = 512
TIME_BUDGET_MINUTES = 27  # same margin as notebook 20 -- leave time to save a checkpoint before a 30-min job gets killed
CHECKPOINT_SEGMENT_BATCHES = 200  # save a chunk file every 200 batches (~3,200 rows at batch_size=16)

CHECKPOINT_PATH = RESULT_DIR / "company_embeddings_checkpoint.npy"  # legacy path, migrated below if found
CHUNKS_DIR = RESULT_DIR / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_PATH = RESULT_DIR / "company_embeddings.npy"
TIME_LOG_PATH = RESULT_DIR / "encode_time_seconds.txt"
prior_encode_secs = float(TIME_LOG_PATH.read_text()) if TIME_LOG_PATH.exists() else 0.0


def atomic_save_npy(arr, path):
    """Write to a temp file then atomically rename -- a plain np.save() left a truncated,
    corrupted checkpoint here once when a job was killed mid-write. Same principle as notebook 54's robust_save."""
    p = Path(path)
    tmp_path = p.with_suffix(".tmp" + p.suffix)
    np.save(tmp_path, arr)
    os.replace(tmp_path, path)


def get_chunk_files():
    return sorted(CHUNKS_DIR.glob("chunk_*.npy"), key=lambda p: int(p.stem.split("_")[1]))


def last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

print("[Encode] Loading Linq-Embed-Mistral (fp16)...")
t0 = time.time()
REPO = "Linq-AI-Research/Linq-Embed-Mistral"
try:
    tokenizer = AutoTokenizer.from_pretrained(REPO, local_files_only=True)
    model = AutoModel.from_pretrained(REPO, torch_dtype=torch.float16, device_map=DEVICE, local_files_only=True)
    print("[Encode] Loaded from local cache -- skipped Hugging Face Hub network calls")
except Exception:
    print("[Encode] Not fully cached locally yet -- loading with network access (this will be slower)")
    tokenizer = AutoTokenizer.from_pretrained(REPO)
    model = AutoModel.from_pretrained(REPO, torch_dtype=torch.float16, device_map=DEVICE)
model.eval()
print(f"[Encode] Model loaded in {(time.time()-t0)/60:.1f} minutes")


@torch.no_grad()
def encode_batch(texts, start=0, batch_size=16):
    """Saves a small chunk file every CHECKPOINT_SEGMENT_BATCHES batches (or at the time budget, or
    at the very end), instead of rewriting one ever-growing checkpoint every time -- the previous
    version rewrote the full accumulated array on every save, which is exactly what triggered a
    bwUniCluster high-I/O warning (746GB written against only 43GB of actual data, a ~17x rewrite
    ratio). This also keeps peak memory bounded to one segment instead of the whole run so far."""
    segment_embs = []
    segment_start = start
    for i in range(start, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_dict = tokenizer(batch, max_length=MAX_LENGTH, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
        outputs = model(**batch_dict)
        embs = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
        embs = F.normalize(embs, p=2, dim=1)
        segment_embs.append(embs.cpu().float().numpy())

        batch_num = (i - start) // batch_size + 1
        current_pos = min(i + batch_size, len(texts))
        if batch_num % 50 == 0:
            print(f"[Encode]   {current_pos:,}/{len(texts):,} companies encoded...")

        is_segment_boundary = batch_num % CHECKPOINT_SEGMENT_BATCHES == 0
        is_time_up = (time.time() - SCRIPT_START) / 60 > TIME_BUDGET_MINUTES
        is_last_batch = current_pos >= len(texts)

        if is_segment_boundary or is_time_up or is_last_batch:
            atomic_save_npy(np.concatenate(segment_embs, axis=0), CHUNKS_DIR / f"chunk_{segment_start:09d}.npy")
            segment_embs = []
            segment_start = current_pos
            if is_time_up and not is_last_batch:
                TIME_LOG_PATH.write_text(str(prior_encode_secs + time.time() - encode_t0))
                print(f"[Encode] Time budget ({TIME_BUDGET_MINUTES} min) reached at {current_pos:,}/{len(texts):,} -- chunk saved, resubmit run.sh to continue")
                sys.exit(0)


if FINAL_PATH.exists() and np.load(FINAL_PATH, mmap_mode="r").shape[0] == len(rich_texts):
    print("[Encode] Final embeddings already on disk -- skipping corpus encoding")
    embeddings = np.load(FINAL_PATH).astype("float32")
    ENCODE_TIME = prior_encode_secs
else:
    chunk_files = get_chunk_files()
    start_idx = sum(np.load(f, mmap_mode="r").shape[0] for f in chunk_files)

    # One-time migration of a legacy single-file checkpoint into the new per-chunk format.
    if CHECKPOINT_PATH.exists() and start_idx == 0:
        legacy = np.load(CHECKPOINT_PATH)
        print(f"[Encode] Migrating legacy checkpoint ({legacy.shape[0]:,} rows) into per-chunk format...")
        atomic_save_npy(legacy, CHUNKS_DIR / f"chunk_{0:09d}.npy")
        start_idx = legacy.shape[0]
        CHECKPOINT_PATH.unlink()

    if start_idx:
        print(f"[Encode] Resuming -- {start_idx:,}/{len(rich_texts):,} companies already encoded ({prior_encode_secs/60:.1f} min spent so far)")

    print("[Encode] Encoding all companies (no instruction prefix on document side)...")
    encode_t0 = time.time()
    encode_batch(rich_texts, start=start_idx, batch_size=16)  # exits early via sys.exit if time budget hit; saves chunks directly
    ENCODE_TIME = prior_encode_secs + (time.time() - encode_t0)

    chunk_files = get_chunk_files()
    total_done = sum(np.load(f, mmap_mode="r").shape[0] for f in chunk_files)
    if total_done == len(rich_texts):
        embeddings = np.concatenate([np.load(f) for f in chunk_files], axis=0)
        atomic_save_npy(embeddings, FINAL_PATH)
        for f in chunk_files:
            f.unlink()
        TIME_LOG_PATH.write_text(str(ENCODE_TIME))
        print(f"[Encode] Done in {ENCODE_TIME/60:.1f} minutes total (across all resumed runs)")
        print(f"[Encode] Embeddings shape : {embeddings.shape}")
        print(f"[Encode] Saved -> {FINAL_PATH}")
    else:
        print(f"[Encode] Not yet complete: {total_done:,}/{len(rich_texts):,} -- resubmit to continue")